# Diagnóstico de dados e protocolo de validação — FOG / Whirlpool (G03)

**Para que serve este notebook.** Antes de gerar qualquer dado novo (sintético, FEM recriado, o que for),
precisamos responder uma pergunta: *o modelo atual está limitado por falta de amostras ou por falta de
informação nas features?* São problemas diferentes e têm soluções opostas. Gerar dados resolve o primeiro
e não faz nada pelo segundo.

Este notebook mede isso. Ele roda em cima de `../data/dataset_preprocessado.csv` e não altera nenhum arquivo.

**Ordem das seções**

| # | Seção | Pergunta que responde |
|---|---|---|
| 1 | Estrutura dos dados | Quantas amostras independentes existem de verdade? |
| 2 | `GroupKFold` — o conceito | Por que a validação atual superestima o modelo |
| 3 | Medindo o vazamento | Quanto a métrica cai quando validamos direito |
| 4 | Configurações distintas | Quantos pontos de projeto o modelo consegue diferenciar |
| 5 | Piso de ruído | Qual o melhor RMSE fisicamente possível com estas features |
| 6 | A coluna `reason` | Onde está a informação que falta |
| 7 | Extração de fatores | Quanto se ganha codificando ela |
| 8 | Harness de avaliação | Como testar dado sintético quando ele existir |

In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, re
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, GroupKFold, LeaveOneGroupOut, cross_val_predict
from sklearn.metrics import mean_absolute_error, r2_score

RANDOM_STATE = 0
N_TREES = 150

df = pd.read_csv('../data/dataset_preprocessado.csv')
df.columns = [c.strip().replace('﻿', '') for c in df.columns]

FATORES = ['blower_direction', 'housing', 'board_position', 'run_case',
           'losses', 'deflector_hole', 'material_coil_support']
ALVO = 'temp'
GRUPO = 'run_id'

print('linhas:', len(df))
print('runs  :', df[GRUPO].nunique())
print('TCs   :', df['tc_number'].nunique())
df.head(3)

linhas: 4065
runs  : 56
TCs   : 77


,run_id,tc_number,tc_name,key_area,active_limit,temp,reason,blower_direction,housing,board_position,run_case,losses,deflector_hole,material_coil_support
0,Run 1,1,"Door Lock Motor Ambient 1/2"" Below",Mainback,155,99.35,Não informado,To back,Open,Center,Pyro,No losses,Open,Aluminium
1,Run 1,2,Door Lock Microswitch Body,Mainback,136,114.72,Não informado,To back,Open,Center,Pyro,No losses,Open,Aluminium
2,Run 1,3,Convect Motor Winding,Mainback,165,89.93,Não informado,To back,Open,Center,Pyro,No losses,Open,Aluminium


## 1. Estrutura dos dados: 4.065 linhas, mas não 4.065 amostras

O arquivo tem ~4.065 linhas. Isso **não** significa 4.065 observações independentes.

Cada `run_id` é **uma** execução do experimento, com **uma** configuração de hardware. Dentro dela,
cada linha é um termopar diferente medindo um ponto diferente do mesmo forno, no mesmo instante,
sob a mesma configuração.

Ou seja: as ~73 linhas de uma run compartilham todas as variáveis de entrada. Elas variam só em
*onde* o termopar está. São medições correlacionadas do mesmo evento, não repetições do experimento.

A unidade estatística real é a **run**. `n = 56`.

In [2]:
por_run = df.groupby(GRUPO).size()
print('linhas por run — min %d, mediana %d, max %d' % (por_run.min(), por_run.median(), por_run.max()))

# Prova de que os fatores são constantes dentro de cada run:
variacao = df.groupby(GRUPO)[FATORES].nunique().max()
print()
print('valores distintos de cada fator DENTRO de uma mesma run (1 = constante):')
print(variacao.to_string())

linhas por run — min 45, mediana 74, max 75

valores distintos de cada fator DENTRO de uma mesma run (1 = constante):
blower_direction         1
housing                  1
board_position           1
run_case                 1
losses                   1
deflector_hole           1
material_coil_support    1


## 2. O conceito: o que é `GroupKFold` e por que ele importa aqui

### Validação cruzada, em uma frase

Para estimar o erro de um modelo em dados que ele nunca viu, separamos parte dos dados,
treinamos no resto, e medimos o erro na parte separada. A validação cruzada (*cross-validation*)
repete isso k vezes, rodando a parte separada por todo o dataset, e junta os erros.

### `KFold` — o jeito padrão, e por que ele quebra aqui

`KFold(shuffle=True)` embaralha as **linhas** e corta em k pedaços.

Com os nossos dados, isso significa que a Run 27 pode ter 60 linhas no treino e 13 no teste.
O modelo vê, durante o treino, o resultado de 60 termopares da Run 27 — e depois é testado
em outros 13 termopares *da mesma run*, com exatamente a mesma configuração de entrada.

Ele não precisa ter aprendido física nenhuma para acertar. Basta ter memorizado
"na Run 27 as coisas ficaram quentes". Isso é **vazamento** (*data leakage*): informação do
conjunto de teste entrou no treino por uma porta lateral.

A métrica que sai daí responde à pergunta errada — *"o modelo interpola entre termopares de uma
run que ele já viu?"* — quando a pergunta do projeto é *"o modelo prevê uma configuração nova?"*.

### `GroupKFold` — o conserto

`GroupKFold` recebe um vetor `groups` e garante que **todas as linhas de um mesmo grupo caem
sempre do mesmo lado do corte**. Passando `groups = run_id`, uma run está inteira no treino
ou inteira no teste — nunca dividida.

Agora o teste simula o cenário real: chega uma configuração de hardware que nunca foi testada,
e o modelo precisa prever a temperatura em cada ponto.

### `LeaveOneGroupOut` — a versão extrema

Caso particular com k = número de grupos: treina em 55 runs, testa em 1, repete 56 vezes.
É o mais honesto e o mais caro. Com n=56 vale a pena, e é o que deve ir para a WAD.

### Resumindo

| | O que embaralha | O que a métrica mede |
|---|---|---|
| `KFold(shuffle=True)` | linhas soltas | interpolação dentro de runs conhecidas (**otimista**) |
| `GroupKFold(groups=run_id)` | runs inteiras | generalização para configuração nova (**honesto**) |
| `LeaveOneGroupOut` | uma run por vez | o mesmo, com o máximo de dados de treino |

### 2.1 Visualizando a diferença

O gráfico abaixo não é decorativo: ele mostra, para as 10 primeiras runs, quais linhas o
`KFold` manda para o teste (espalhadas por todas as runs) versus o `GroupKFold`
(runs inteiras, blocos sólidos).

In [3]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

sub = df[df[GRUPO].isin(df[GRUPO].unique()[:10])].reset_index(drop=True)
g_sub = sub[GRUPO].values
idx = np.arange(len(sub))

fold_kf = np.zeros(len(sub), int)
for i, (_, te) in enumerate(KFold(5, shuffle=True, random_state=0).split(sub)):
    fold_kf[te] = i
fold_gk = np.zeros(len(sub), int)
for i, (_, te) in enumerate(GroupKFold(5).split(sub, groups=g_sub)):
    fold_gk[te] = i

fig, axes = plt.subplots(2, 1, figsize=(11, 3.2), sharex=True)
for ax, fold, titulo in zip(axes, [fold_kf, fold_gk],
                            ['KFold(shuffle=True) — folds picotam as runs',
                             'GroupKFold(groups=run_id) — cada run inteira em um fold']):
    ax.scatter(idx, np.zeros(len(sub)), c=fold, cmap='tab10', marker='|', s=400, linewidths=2)
    ax.set_yticks([]); ax.set_title(titulo, fontsize=10, loc='left')
for r in sub[GRUPO].unique():
    pos = idx[sub[GRUPO] == r]
    for ax in axes:
        ax.axvline(pos[0] - 0.5, color='k', lw=0.6, alpha=0.4)
axes[1].set_xlabel('linha (linhas verticais pretas = fronteira entre runs)')
plt.tight_layout(); plt.show()

## 3. Medindo o vazamento

Mesmo modelo, mesmas features, só muda o esquema de validação.

In [4]:
def montar_X(dados, extras=None):
    X = pd.get_dummies(dados[FATORES + ['key_area', 'tc_name']].astype(str))
    X['active_limit'] = dados['active_limit'].values
    X['tc_number'] = dados['tc_number'].values
    if extras is not None:
        X = pd.concat([X.reset_index(drop=True), extras.reset_index(drop=True)], axis=1)
    return X.astype(float)

def avaliar(X, y, cv, groups=None, nome=''):
    modelo = RandomForestRegressor(n_estimators=N_TREES, random_state=RANDOM_STATE, n_jobs=-1)
    pred = cross_val_predict(modelo, X.values, y, cv=cv, groups=groups, n_jobs=1)
    rmse = float(np.sqrt(((pred - y) ** 2).mean()))
    return {'esquema': nome, 'RMSE': rmse, 'MAE': mean_absolute_error(y, pred), 'R2': r2_score(y, pred)}

X = montar_X(df)
y = df[ALVO].values
g = df[GRUPO].values

resultados = [
    avaliar(X, y, KFold(5, shuffle=True, random_state=RANDOM_STATE), None, 'KFold aleatorio (VAZADO)'),
    avaliar(X, y, GroupKFold(5), g, 'GroupKFold por run_id'),
    avaliar(X, y, LeaveOneGroupOut(), g, 'LeaveOneRunOut'),
]

media_tc = df.groupby('tc_number')[ALVO].transform('mean').values
resultados.append({'esquema': 'baseline: media do TC', 'RMSE': float(np.sqrt(((media_tc - y) ** 2).mean())),
                   'MAE': mean_absolute_error(y, media_tc), 'R2': r2_score(y, media_tc)})

tabela = pd.DataFrame(resultados).set_index('esquema').round(3)
tabela

,RMSE,MAE,R2
esquema,,,
KFold aleatorio (VAZADO),7.731,4.281,0.946
GroupKFold por run_id,8.084,4.711,0.941
LeaveOneRunOut,8.272,4.795,0.938
baseline: media do TC,16.420,9.487,0.757


**Como ler.** A diferença entre a linha 1 e a linha 2 é o tamanho do vazamento. Se ela for grande,
a métrica reportada até agora estava inflada e o número honesto é o do `GroupKFold`.

A quarta linha é o piso de comparação: um "modelo" que ignora toda a configuração e chuta a média
histórica daquele termopar. Qualquer modelo precisa bater isso com folga para estar aprendendo
alguma coisa sobre o hardware, e não só decorando quais sensores são quentes.

**A partir daqui, use `GroupKFold` ou `LeaveOneGroupOut` em tudo.** Nunca mais `KFold` simples
neste dataset.